# SAU-Style CNO-FNO (Axial Attention) vs WHNO: Spectral Basis Comparison

This trains two operator-learning models on the same geometry: `CNOFNOHybrid` with axial
self-attention (`use_attention=True`, the SAU-FNO-style architecture already in this repo)
against `WHNO3d` (Walsh-Hadamard Neural Operator, a Fourier-basis alternative implemented
specifically because Walsh-Hadamard's piecewise-constant basis functions don't suffer the
Gibbs ringing that Fourier truncation shows at sharp material-conductivity discontinuities;
see `src/fno/whno.py` for the validated 1D step-function demo).

Both models train with the (now harmonic-mean-corrected) whole-volume PDE loss and the
interface-isolated flux-continuity loss (`src/fno/physics.py`), and are compared via:
1. Accuracy, speed, and parameter count
2. Spectral mode-importance XAI (`src/fno/interpret.py`): reads the learned weight tensors
   directly, no forward passes needed
3. Interface-distance-bucketed error: does WHNO's accuracy advantage (if any) concentrate
   specifically near material boundaries, as the Gibbs-ringing argument predicts?

## Kaggle Dataset Setup

1. Source code: upload `src/` (slug suggestion: `thermo-pinn-src`)
2. 3D-ICE training data: upload `.npz` files from `data/3d-ice/` (slug suggestion: `3dice-thermal-data`)

Default geometry is `geometry1` for fast iteration. For a stronger interface-discontinuity
story, switch `GEOM_NAME` to `geometry6` (11 layers, 42x14mm footprint); much slower to
train.

Expected time: about 15-25 min/model on a T4 at 400 epochs for geometry1 (2 models = about
30-50 min total).


In [ ]:
import subprocess, sys

# Install any missing packages (PyYAML is the only non-standard dep)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml'], check=True)

import os
import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
import sys
from pathlib import Path

# ── Kaggle dataset paths ────────────────────────────────────────────────────
# Adjust these slugs to match the datasets you attached to this notebook.
SRC_DATASET   = 'thermo-pinn-src'    # dataset containing the src/ folder
DATA_DATASET  = '3dice-thermal-data' # dataset containing the .npz files

SRC_ROOT  = Path(f'/kaggle/input/{SRC_DATASET}')
DATA_ROOT = Path(f'/kaggle/input/{DATA_DATASET}')
OUT_DIR   = Path('/kaggle/working/checkpoints/sau_cnofno_vs_whno')

# Add source root to path so we can import src.*
sys.path.insert(0, str(SRC_ROOT))

# Verify
assert SRC_ROOT.exists(),  f'Source dataset not found at {SRC_ROOT}. Check SRC_DATASET slug.'
assert DATA_ROOT.exists(), f'Data dataset not found at {DATA_ROOT}. Check DATA_DATASET slug.'
print('Source root:', SRC_ROOT)
print('Data root:  ', DATA_ROOT)

In [ ]:
import logging, time
import numpy as np
import torch
import matplotlib.pyplot as plt

from src.core.geometry_builders import get_geometry_by_name
from src.pinn.data_loader import NormStats, compute_norm_stats
from src.fno.model import build_cno_fno
from src.fno.whno import build_whno
from src.fno.data_loader import FNODataset
from src.fno.trainer import FNOTrainer
from src.fno.interpret import model_mode_importance, marginal_importance, plot_mode_importance_comparison

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)-8s %(name)s: %(message)s',
    datefmt='%H:%M:%S',
)
print('Imports OK')

In [ ]:
GEOM_NAME    = 'geometry1'   # alt: 'geometry6' for a stronger interface-discontinuity story (11 layers) -- much slower
CHANNELS     = 32
N_BLOCKS     = 4
N_HEADS      = 4             # CNO-FNO axial attention heads; must divide CHANNELS evenly
MODES        = (16, 16, 12)  # WHNO's low-sequency mode counts (same shape convention as FNO)
EPOCHS       = 400           # cno-fno default; whno uses the same epoch budget for a fair comparison
BATCH_SIZE   = 4
LR           = 1e-3
PDE_WEIGHT   = 0.1           # whole-volume finite-difference PDE loss (harmonic-mean face conductivity)
FLUX_WEIGHT  = 0.5           # interface-isolated flux-continuity loss (src/fno/physics.py)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {DEVICE}')
print(f'Config: channels={CHANNELS} blocks={N_BLOCKS} epochs={EPOCHS} pde_weight={PDE_WEIGHT} flux_weight={FLUX_WEIGHT}')

In [ ]:
geometry = get_geometry_by_name(GEOM_NAME)
geometries = {GEOM_NAME: geometry}
grid_shape = geometry.mesh_resolution
print(geometry.summary())
print('Grid shape:', grid_shape)

In [ ]:
def collect_files(data_dir: Path, geom: str, split: str):
    files = sorted(data_dir.rglob(f'{geom}_{split}_*.npz'))
    if not files:
        files = sorted(data_dir.glob(f'{geom}_{split}_*.npz'))
    return files

In [ ]:
train_files = collect_files(DATA_ROOT, GEOM_NAME, 'train')
test_files  = collect_files(DATA_ROOT, GEOM_NAME, 'test')

assert train_files, f'No training files found in {DATA_ROOT}. Check DATA_DATASET slug and file naming.'
print(f'Training files : {len(train_files)}')
print(f'Test files     : {len(test_files)}')

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
norm_path = OUT_DIR / 'norm_stats.json'

if norm_path.exists():
    norm_stats = NormStats.load(norm_path)
    print('Loaded existing norm stats from', norm_path)
else:
    print('Computing norm stats over', len(train_files), 'training files...')
    norm_stats = compute_norm_stats(train_files, geometries)
    norm_stats.save(norm_path)
    print('Saved norm stats to', norm_path)

print(f'T range: [{norm_stats.T_min:.1f}, {norm_stats.T_max:.1f}] K')

In [ ]:
print('Loading training dataset...')
train_dataset = FNODataset(train_files, norm_stats, grid_shape)

val_files = test_files if test_files else train_files[-3:]
print('Loading validation dataset...')
val_dataset = FNODataset(val_files, norm_stats, grid_shape)

print(f'Train: {len(train_dataset)} scenarios')
print(f'Val  : {len(val_dataset)} scenarios')

In [ ]:
cno_fno = build_cno_fno(
    grid_shape=grid_shape, ch=CHANNELS, n_fno_blocks=N_BLOCKS,
    use_attention=True, n_heads=N_HEADS, device=DEVICE,
)
whno = build_whno(
    grid_shape=grid_shape, modes=MODES, hidden_ch=CHANNELS, n_blocks=N_BLOCKS, device=DEVICE,
)

print(f'CNO-FNO (SAU-style, attention=True): {cno_fno.n_parameters:,} params')
print(f'WHNO: {whno.n_parameters:,} params')

In [ ]:
out_cno = OUT_DIR / 'cno_fno_attn't0 = time.time()trainer_cno = FNOTrainer(    model=cno_fno, norm_stats=norm_stats, train_data=train_dataset, val_data=val_dataset,    output_dir=out_cno, batch_size=BATCH_SIZE, epochs=EPOCHS, lr=LR, device=DEVICE,    geometry_name=f'{GEOM_NAME}_cno_fno_attn', pde_weight=PDE_WEIGHT, flux_weight=FLUX_WEIGHT, geometry=geometry,        use_amp=False,  # cuFFT half-precision needs power-of-two grids; ours never are    )ckpt_cno = trainer_cno.train()time_cno = time.time() - t0print(f'CNO-FNO training time: {time_cno/60:.1f} min')

In [ ]:
out_whno = OUT_DIR / 'whno't0 = time.time()trainer_whno = FNOTrainer(    model=whno, norm_stats=norm_stats, train_data=train_dataset, val_data=val_dataset,    output_dir=out_whno, batch_size=BATCH_SIZE, epochs=EPOCHS, lr=LR, device=DEVICE,    geometry_name=f'{GEOM_NAME}_whno', pde_weight=PDE_WEIGHT, flux_weight=FLUX_WEIGHT, geometry=geometry,        use_amp=False,  # cuFFT half-precision needs power-of-two grids; ours never are    )ckpt_whno = trainer_whno.train()time_whno = time.time() - t0print(f'WHNO training time: {time_whno/60:.1f} min')

In [ ]:
print(f'{"Model":<20} {"Params":>12} {"Best val MAE (K)":>18} {"Train time (min)":>18}')
print('-' * 70)
print(f'{"CNO-FNO (SAU)":<20} {cno_fno.n_parameters:>12,} {trainer_cno.best_val_mae:>18.3f} {time_cno/60:>18.1f}')
print(f'{"WHNO":<20} {whno.n_parameters:>12,} {trainer_whno.best_val_mae:>18.3f} {time_whno/60:>18.1f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(trainer_cno.history['epoch'], trainer_cno.history['val_mae_K'], label='CNO-FNO (SAU)', color='steelblue')
ax.plot(trainer_whno.history['epoch'], trainer_whno.history['val_mae_K'], label='WHNO', color='tomato')
ax.set_xlabel('Epoch'); ax.set_ylabel('Val MAE (K)')
ax.set_title(f'{GEOM_NAME}: CNO-FNO (SAU-style) vs WHNO'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
imp_cno = model_mode_importance(cno_fno)
imp_whno = model_mode_importance(whno)
n_blocks_compare = min(len(imp_cno), len(imp_whno))
print(f'Comparing {n_blocks_compare} spectral blocks')

for b in range(n_blocks_compare):
    marg_cno  = marginal_importance(imp_cno[b],  axis=2)  # z-axis
    marg_whno = marginal_importance(imp_whno[b], axis=2)
    out_path = OUT_DIR / f'mode_importance_block{b}.png'
    plot_mode_importance_comparison(
        {'CNO-FNO (SAU)': marg_cno, 'WHNO': marg_whno}, out_path,
        title=f'Block {b}: z-axis spectral mode importance ({GEOM_NAME})',
    )
    print(f'Block {b}: saved {out_path.name}')

from IPython.display import Image, display
display(Image(str(OUT_DIR / 'mode_importance_block0.png')))

In [ ]:
def interface_distance_bucketed_mae(model, val_item, geometry, norm_stats, n_buckets=6):
    """Bucket |T_pred-T_true| by distance (µm) to nearest material interface z-plane."""
    Q = val_item['Q_norm'].unsqueeze(0).to(DEVICE)
    L = val_item['layer_id_norm'].unsqueeze(0).to(DEVICE)
    T_true = val_item['T_norm'].unsqueeze(0).to(DEVICE)
    htc = val_item['htc_norm'].to(DEVICE)
    tamb = val_item['t_amb_norm'].to(DEVICE)
    tsv = val_item['tsv_frac'].to(DEVICE)

    with torch.no_grad():
        T_pred = model(Q, L, htc, tamb, tsv)

    T_range = norm_stats.T_max - norm_stats.T_min
    err_K = ((T_pred - T_true) * T_range).abs().squeeze(0).cpu().numpy()  # (nx,ny,nz)

    nx, ny, nz = err_K.shape
    total_h = geometry.get_total_height()
    z_centers = (np.arange(nz) + 0.5) / nz * total_h  # µm

    interface_zs = np.array(sorted(set(l.z_top for l in geometry.layers[:-1])))
    dist_to_interface = np.array([np.min(np.abs(z - interface_zs)) for z in z_centers])  # (nz,)

    max_dist = dist_to_interface.max()
    bucket_edges = np.linspace(0, max_dist, n_buckets + 1)
    bucket_mae = []
    for i in range(n_buckets):
        mask_z = (dist_to_interface >= bucket_edges[i]) & (dist_to_interface < bucket_edges[i + 1])
        bucket_mae.append(err_K[:, :, mask_z].mean() if mask_z.sum() > 0 else np.nan)
    bucket_centers = (bucket_edges[:-1] + bucket_edges[1:]) / 2
    return bucket_centers, np.array(bucket_mae)

val_item0 = val_dataset.items[0]
bc, mae_cno  = interface_distance_bucketed_mae(cno_fno, val_item0, geometry, norm_stats)
_,  mae_whno = interface_distance_bucketed_mae(whno,    val_item0, geometry, norm_stats)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(bc, mae_cno, marker='o', label='CNO-FNO (SAU)', color='steelblue')
ax.plot(bc, mae_whno, marker='o', label='WHNO', color='tomato')
ax.set_xlabel('Distance to nearest material interface (µm)')
ax.set_ylabel('MAE (K)')
ax.set_title(f'{GEOM_NAME}: error vs distance-to-interface\n(validation scenario: {val_item0.get("name", "?")})')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'interface_distance_mae.png', dpi=150, bbox_inches='tight')
plt.show()

print('If WHNO shows lower MAE at small distance-to-interface than CNO-FNO,')
print('that is direct evidence WHNO handles the material discontinuities better.')

In [ ]:
import json

summary = {
    'geometry': GEOM_NAME,
    'grid_shape': list(grid_shape),
    'cno_fno': {
        'params': cno_fno.n_parameters,
        'best_val_mae_K': trainer_cno.best_val_mae,
        'train_time_min': time_cno / 60,
    },
    'whno': {
        'params': whno.n_parameters,
        'best_val_mae_K': trainer_whno.best_val_mae,
        'train_time_min': time_whno / 60,
    },
    'interface_distance_mae': {
        'bucket_centers_um': bc.tolist(),
        'cno_fno': mae_cno.tolist(),
        'whno': mae_whno.tolist(),
    },
    'checkpoints': {'cno_fno': str(ckpt_cno), 'whno': str(ckpt_whno)},
}

with open(OUT_DIR / 'sau_cnofno_vs_whno_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))